el resultado al que debo llegar es

Puntualidad -> 0.88 Mayo26

In [1]:
import pandas as pd
import holidays
import numpy as np
from pathlib import Path

In [8]:
ROOT = Path.cwd().resolve().parent
DATA = ROOT / "data"

a5_path = DATA / "toptur/POT_V_VALPARAISOUN07_UN07_NORMAL_2026_2_A5__2.xlsx"
expediciones_path = DATA / "toptur/Mayo26/expediciones.xls"

In [3]:
SALIDA = ROOT / "salida"

empresa = "toptur"
mes = "Mayo"
anio = "26"

In [4]:
def a_segundos(valor):
    """Convierte un valor de hora a segs"""
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, pd.Timedelta):
        return valor.total_seconds()
    if hasattr(valor, "hour"): #datetime.time
        return valor.hour * 3600 + valor.minute*60 + valor.second
    h,m,s = map(int, str(valor).split(":"))
    return h*3600 + m*60 + s

In [5]:
def calcular_ip_un_tpp(tpp, ipp_ant, ipp_post, lpo_disponible):
    niveles = [
        (tpp - ipp_ant/12,  tpp + ipp_post/6,  1.0),
        (tpp - ipp_ant/6,   tpp - ipp_ant/12,  0.75),
        (tpp + ipp_post/6,  tpp + ipp_post/3,  0.75),
        (tpp - ipp_ant/4,   tpp - ipp_ant/6,   0.5),
        (tpp + ipp_post/3,  tpp + ipp_post/2,  0.5),
        (tpp - ipp_ant/3,   tpp - ipp_ant/4,   0.25),
        (tpp + ipp_post/2,  tpp + (2/3)*ipp_post, 0.25),
    ]
    for lo, hi, valor in niveles:
        for bus in lpo_disponible:
            tpo = bus["TPO_seg"] # Extraemos los segundos del diccionario
            if lo <= tpo <= hi:
                return valor, bus # Retornamos el diccionario completo
                
    return 0.0, None

In [6]:
feriados_cl = holidays.Chile()
def clasificar_tipo_dia(fecha):
    fecha = pd.Timestamp(fecha).date()
    if fecha in feriados_cl:
        return "DF"
    dow = pd.Timestamp(fecha).dayofweek  # 0=lunes ... 6=domingo
    if dow == 6:
        return "DF"
    elif dow == 5:
        return "DS"
    else:
        return "DL"

In [9]:
df_expediciones = pd.read_html(expediciones_path)[0]

In [ ]:
df_expediciones.to_csv("./expediciones.csv")

In [7]:
mapeo_sentido = {"Ida": 0, "Reg": 1, "I": 0, "V": 1}

df_expediciones["Sentido"] = df_expediciones["Sentido"].replace(mapeo_sentido)
df_expediciones["Estado"] = df_expediciones["Estado"].str.strip().str.lower()
df_expediciones = df_expediciones[df_expediciones["Estado"] == "valida"].copy()

df_expediciones["fecha"] = pd.to_datetime(df_expediciones["Inicio Expedicion"]).dt.date
df_expediciones["tipo_dia"] = df_expediciones["fecha"].apply(clasificar_tipo_dia)

In [8]:
df_a5 = pd.read_excel(a5_path, sheet_name="LPP", skiprows=10, header=0)
campos = {
    "Correlativo Punto\nde Control": "correlativo_pc",
    "Intervalo Anterior\n(IPPdk-1)": "IPP_anterior",
    "Hora de Pasada Programada\n(TPPdk)": "TPP",
    "Intervalo Posterior\n(IPPdk)": "IPP_posterior",
    "Tipo de Día": "tipo_dia"
}
df_a5 = df_a5.rename(columns=campos)

for col in ["IPP_anterior", "TPP", "IPP_posterior"]:
    df_a5[col + "_seg"] = df_a5[col].apply(a_segundos)

In [34]:
df_a5

,UN,Servicio,Sentido,correlativo_pc,IPP_anterior,TPP,IPP_posterior,tipo_dia,IPP_anterior_seg,TPP_seg,IPP_posterior_seg
0,UN07,701,0,1,00:30:00,6:05:00,00:07:30,DL,1800,21900,450


In [35]:
df_expediciones.Sentido

2        0
3        0
4        0
5        0
6        1
        ..
20859    1
20860    1
20861    0
20862    1
20863    1
Name: Sentido, Length: 19384, dtype: object

In [9]:
columnas_pc = [c for c in df_expediciones.columns if str(c).isdigit()] #PC
df_lpo = df_expediciones.melt(
    id_vars=["Servicio","Sentido","fecha","Inicio Expedicion","Estado","Bus","Periodo"],
    value_vars=columnas_pc,
    var_name="correlativo_pc",
    value_name="TPO"
)

df_lpo["correlativo_pc"] = df_lpo["correlativo_pc"].astype(int)
df_lpo["TPO_seg"] = df_lpo["TPO"].apply(a_segundos)
df_lpo = df_lpo[(df_lpo["Estado"] == "valida") & df_lpo["TPO_seg"].notna()]

In [48]:
df_lpo

,Servicio,Sentido,fecha,Inicio Expedicion,Estado,Bus,Periodo,correlativo_pc,TPO,TPO_seg
0,701,0,2026-05-01,2026-05-01 06:59:52,valida,116/FRDH12,6,1,06:59:52,25192.0
1,705,0,2026-05-01,2026-05-01 07:00:34,valida,153/GWXF15,7,1,07:00:34,25234.0
2,703,0,2026-05-01,2026-05-01 07:01:11,valida,219/DHYV15,7,1,07:01:11,25271.0
3,702,0,2026-05-01,2026-05-01 07:01:43,valida,107/FXRL75,7,1,07:01:43,25303.0
4,702,1,2026-05-01,2026-05-01 07:16:40,valida,108/GWXF20,7,1,07:16:40,26200.0
...,...,...,...,...,...,...,...,...,...,...
290740,705,0,2026-05-31,2026-05-31 19:44:59,valida,148/FTZP86,19,15,20:36:07,74167.0
290743,705,1,2026-05-31,2026-05-31 19:52:43,valida,28/GWWW38,19,15,20:40:19,74419.0
290750,705,1,2026-05-31,2026-05-31 20:14:53,valida,90/FSBF91,20,15,20:58:15,75495.0
290753,705,1,2026-05-31,2026-05-31 20:27:12,valida,166/GXWT51,20,15,21:11:17,76277.0


In [10]:
servicios_sentido_a5 = list(df_a5[["Servicio", "Sentido"]].drop_duplicates().itertuples(index=False, name=None))

servicios_sentido_a5

[(701, 0)]

In [13]:
resultados = []

for fecha in sorted(df_expediciones["fecha"].unique()):
    tipo_dia_actual = clasificar_tipo_dia(fecha) 
    lpo_del_dia = df_lpo[df_lpo["fecha"] == fecha]
    
    for (servicio, sentido) in servicios_sentido_a5:
        # Como ahora clasificar_tipo_dia retorna "DL", este filtro funcionará y no quedará vacío.
        lpp_grupo = df_a5[
            (df_a5.Servicio == servicio) &
            (df_a5.Sentido == sentido) &
            (df_a5.tipo_dia == tipo_dia_actual)
        ].sort_values("correlativo_pc")
        
        if lpp_grupo.empty:
            continue  
            
        for pc in lpp_grupo["correlativo_pc"].unique():
            lpp_pc = lpp_grupo[lpp_grupo.correlativo_pc == pc].sort_values("TPP_seg")
            
            # 1. Filtramos y convertimos a diccionarios en lugar de .tolist()
            lpo_pc = lpo_del_dia[
                (lpo_del_dia.Servicio == servicio) &
                (lpo_del_dia.Sentido == sentido) &
                (lpo_del_dia.correlativo_pc == pc)
            ][["TPO_seg", "Periodo"]].to_dict('records')
            
            # 2. Ordenamos cronológicamente usando la clave "TPO_seg"
            lpo_pc.sort(key=lambda x: x["TPO_seg"])
            
            for _, fila in lpp_pc.iterrows():
                ip_valor, bus_usado = calcular_ip_un_tpp(
                    fila["TPP_seg"], fila["IPP_anterior_seg"], fila["IPP_posterior_seg"], lpo_pc
                )
                
                # 3. Extraemos el periodo del bus que hizo match
                periodo_usado = None
                if bus_usado is not None:
                    periodo_usado = bus_usado["Periodo"]
                    lpo_pc.remove(bus_usado)
                    
                resultados.append({
                    "fecha": fecha,
                    "tipo_dia": tipo_dia_actual,
                    "Servicio": servicio,
                    "Sentido": sentido,
                    "correlativo_pc": pc,
                    "TPP": fila["TPP"],
                    "Periodo": periodo_usado, # <-- ¡NUEVO CAMPO AQUÍ!
                    "IP": ip_valor
                })

df_resultados = pd.DataFrame(resultados)

In [14]:
df_resultados

,fecha,tipo_dia,Servicio,Sentido,correlativo_pc,TPP,Periodo,IP
0,2026-05-04,DL,701,0,1,6:05:00,6,1.00
1,2026-05-05,DL,701,0,1,6:05:00,6,1.00
2,2026-05-06,DL,701,0,1,6:05:00,6,1.00
3,2026-05-07,DL,701,0,1,6:05:00,6,0.75
4,2026-05-08,DL,701,0,1,6:05:00,6,1.00
5,2026-05-11,DL,701,0,1,6:05:00,6,1.00
6,2026-05-12,DL,701,0,1,6:05:00,6,0.75
7,2026-05-13,DL,701,0,1,6:05:00,6,0.75
8,2026-05-14,DL,701,0,1,6:05:00,6,1.00
9,2026-05-15,DL,701,0,1,6:05:00,6,1.00


In [15]:
ip_promedio = df_resultados.IP.mean().round(2)

if ip_promedio < 0.50:
    ip_final = 0.50
elif ip_promedio > 0.90:
    ip_final = 1.0
else:
    ip_final = ip_promedio

print(f"IP_M' = {ip_promedio}  →  IP_M = {ip_final}")

IP_M' = 0.88  →  IP_M = 0.88


In [20]:
# 1. Calcular el promedio general de la variable deseada
ip_promedio = df_resultados["IP"].mean()

# 2. Definir el nombre del archivo de salida
nombre_archivo_excel = SALIDA / f"reporte_IP_{empresa}_{mes}{anio}.xlsx"

# 3. Exportar usando el motor 'openpyxl' para poder modificar celdas después de escribir el DataFrame
with pd.ExcelWriter(nombre_archivo_excel, engine='openpyxl') as writer:
    # Guardamos el DataFrame normal en la pestaña 'Reporte'
    df_resultados.to_excel(writer, sheet_name='Reporte', index=False)
    
    # Obtenemos la hoja de trabajo activa para agregar las filas adicionales
    worksheet = writer.sheets['Reporte']
    
    # Calculamos la fila exacta: Total de datos + 1 fila de encabezado + 2 filas de separación (salto)
    fila_salto = len(df_resultados) + 3
    
    # Escribimos el texto y el valor del promedio en esa fila
    worksheet.cell(row=fila_salto, column=1, value="Promedio IP:")
    worksheet.cell(row=fila_salto, column=2, value=ip_promedio)
    
    # (Opcional) Poner en negrita la celda del resultado para que destaque
    from openpyxl.styles import Font
    worksheet.cell(row=fila_salto, column=1).font = Font(bold=True)
    worksheet.cell(row=fila_salto, column=2).font = Font(bold=True)

print(f"Reporte exportado exitosamente a {nombre_archivo_excel}")

Reporte exportado exitosamente a C:\Users\Raulito\Desktop\NABLA SPA\Codigos\IP\salidas\reporte_IP_toptur_Mayo26.xlsx
